# 12_deploy_serving_endpoint_v3

Update the existing World Bank GEP serving endpoint to the newly registered V3 model. Run this only after Notebook 11 V3 finishes and prints the new model version.


In [0]:
# CELL 1 — Install SDK if needed
%pip install -q "databricks-sdk>=0.102.0"


If `%pip` updates the SDK, restart Python once before continuing.


In [0]:
# CELL 2 — Configuration
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, ServingModelWorkloadType

MODEL_NAME = "worldbank_ai.ai.gep_intelligence_agent"
MODEL_VERSION = "3"  # Change ONLY if Notebook 11 prints a different new version.
ENDPOINT_NAME = "worldbank-gep-intelligence-agent"
SQL_WAREHOUSE_ID = "3d11225dd32e8158"

w = WorkspaceClient()
print("Model:", MODEL_NAME)
print("Version:", MODEL_VERSION)
print("Endpoint:", ENDPOINT_NAME)


In [0]:
# CELL 3 — Confirm endpoint exists and inspect current state
endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
print("Endpoint ID:", endpoint.id)
print("Current state:", endpoint.state)
if endpoint.config:
    print("Active config version:", endpoint.config.config_version)
if endpoint.pending_config:
    print("Pending config version:", endpoint.pending_config.config_version)


In [0]:
# CELL 4 — Define V3 served entity
served_entity_v3 = ServedEntityInput(
    name="gep-intelligence-agent-v3",
    entity_name=MODEL_NAME,
    entity_version=MODEL_VERSION,
    workload_type=ServingModelWorkloadType.CPU,
    workload_size="Small",
    scale_to_zero_enabled=True,
    environment_vars={
        "DATABRICKS_SQL_WAREHOUSE_ID": SQL_WAREHOUSE_ID,
    },
)
print("Prepared served entity:", served_entity_v3.name)


In [0]:
# CELL 5 — Update existing endpoint to V3
print("Updating existing endpoint to model version", MODEL_VERSION)
deployment = w.serving_endpoints.update_config_and_wait(
    name=ENDPOINT_NAME,
    served_entities=[served_entity_v3],
)
print("Update completed")


In [0]:
# CELL 6 — Final READY validation
endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
print("Final state:", endpoint.state)
print("Active config version:", endpoint.config.config_version if endpoint.config else None)

if endpoint.config:
    for entity in endpoint.config.served_entities or []:
        print("Served entity:", entity.name)
        print("Model:", entity.entity_name)
        print("Version:", entity.entity_version)

state_text = str(endpoint.state)
if "READY" not in state_text or "UPDATE_FAILED" in state_text:
    raise RuntimeError(f"Endpoint is not READY: {endpoint.state}")
print("PASS: endpoint is READY on model version", MODEL_VERSION)
